# MHCoT — GoT Preprocessing on Colab

Runs Besta GoT (via local DeepSeek-R1-Distill-Qwen-1.5B) over GSM8K and caches thought-node outputs to Google Drive.

**Runtime:** T4 GPU (free tier is enough). Set via *Runtime → Change runtime type → T4 GPU*.

**Total time:** ~3-5 hours for full GSM8K test split.

## Cell 1 — Install dependencies (one shot, ~3 min)

In [ ]:
# Install a tested set of pins that we know work together.
# Colab's base image already has compatible torch + cuda; we override only what we need.
%pip install -q \
    "transformers>=4.45,<5" \
    "huggingface_hub>=0.24,<0.30" \
    "datasets>=3.0,<4" \
    "accelerate>=0.34,<1" \
    "sentencepiece" \
    "tqdm" \
    "openai<1.0" \
    "graph_of_thoughts==0.0.2"

# Quick sanity check
import torch, transformers, datasets, huggingface_hub
print('torch          ', torch.__version__)
print('CUDA available ', torch.cuda.is_available())
print('GPU            ', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('transformers   ', transformers.__version__)
print('datasets       ', datasets.__version__)
print('huggingface_hub', huggingface_hub.__version__)

# Confirm Besta imports cleanly
from graph_of_thoughts import controller, operations, parser, prompter
print('graph_of_thoughts OK')

## Cell 2 — Mount Google Drive (for persistent output)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/MHCoT'
os.makedirs(f'{PROJECT_DIR}/data/got_cache', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/main', exist_ok=True)
print('project dir ready at', PROJECT_DIR)

## Cell 3 — Upload `got_runner.py` and `preprocess_got.py`

**Option A:** Drag-and-drop these two files from your Mac into the Colab sidebar (folder icon → upload). Then run the cell below to move them to Drive.

**Option B:** Copy-paste the files directly via `%%writefile` magic (see the Option B cell below).

In [ ]:
# Option A: copy uploaded files to Drive
import shutil, os
for f in ['got_runner.py', 'preprocess_got.py']:
    src = f'/content/{f}'
    dst = f'{PROJECT_DIR}/main/{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'copied {f} -> {dst}')
    elif os.path.exists(dst):
        print(f'{f} already in Drive')
    else:
        print(f'MISSING: upload {f} via sidebar first')

## Cell 4 — Smoke test (5 problems, ~10-20 min on T4)

In [ ]:
import sys
sys.path.insert(0, f'{PROJECT_DIR}/main')

import os
os.chdir(PROJECT_DIR)   # so relative data/got_cache paths resolve under Drive

!python {PROJECT_DIR}/main/preprocess_got.py --split test --limit 5

## Cell 5 — Verify smoke output

In [ ]:
import json
path = f'{PROJECT_DIR}/data/got_cache/gsm8k_test.jsonl'
with open(path) as f:
    lines = f.readlines()
print(f'cached: {len(lines)} problems')
rec = json.loads(lines[0])
print('idx       ', rec['idx'])
print('gold      ', rec['gold'])
print('n_nodes   ', len(rec['thought_nodes']))
print('wall_secs ', round(rec['wall_seconds'], 1))
print('---')
print(rec['thought_nodes'][0][:1500])

## Cell 6 — Full test split (only after smoke test looks good)

~3-5 hours on T4. Cache survives session disconnects (resumable).

**Free Colab note:** sessions can idle out after ~90 min. Two options:
- Stay active in the tab and re-run this cell if it disconnects (it resumes).
- Or upgrade to Colab Pro for longer sessions.

In [ ]:
!python {PROJECT_DIR}/main/preprocess_got.py --split test